<a href="https://colab.research.google.com/github/YasirAktas/FLwithHE/blob/dp-merge-main/Clipping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced DP Clipping in HE-Federated Learning


In this pipeline, we perform Federated Learning over non-IID Dirichlet partitions. To protect data privacy, clients encrypt their final model weights using Homomorphic Encryption (Paillier/CKKS) before transmission.

Because Dirichlet partitions create extreme local data skew, gradient norms can explode, destabilizing the global model and ruining Differential Privacy (DP) bounds. To fix this, we utilize three advanced gradient clipping strategies: Fixed, Quantile, and Adaptive.


A known roadblock in secure FL is that an Aggregator cannot calculate quantiles or adapt thresholds over encrypted weights. Our codebase elegantly bypasses this. If you inspect client.py, the train() method passes all clipping parameters directly into apply_dp_sgd(). This means:

The client performs clipping and noise addition locally on plaintext micro-batches.

The client finishes training its local epochs.

Only then does the client encrypt the final state_dict using encryption_context.encrypt().

The Aggregator (in aggregator.py) simply executes a blind homomorphic sum: self.encryption_context.add(acc, part). It never needs to know the norms

**Strategy 1: Fixed Clipping (The DP Baseline)**
Concept: A static, hardcoded limit (--dp_clip_norm) is applied to every gradient during local training.

The 'Why': It guarantees a strict sensitivity bound required for mathematical DP.

The Trade-off: With Dirichlet partitions, picking a single number is dangerous. If set to 1.0, it might destroy the learning signal of a client with highly skewed, high-norm data, while leaving another client entirely untouched.


Execution: Fixed Clipping

In [1]:
#@title  Launch Fixed Clipping DP-FL
#@markdown This executes the baseline fixed clipping strategy on ourr project.

dataset = "cifar10" #@param ["mnist", "cifar10", "ptbxl"]
num_clients = 5 #@param {type:"slider", min:1, max:20, step:1}
dirichlet_alpha = 0.5 #@param {type:"slider", min:0.1, max:5.0, step:0.1}

# DP Settings
dp_clip_norm = 1.0 #@param {type:"slider", min:0.1, max:10.0, step:0.1}
dp_epsilon = 2.0 #@param {type:"number"}
dp_mechanism = "gaussian" #@param ["gaussian", "laplace"]

!python -m src.fl.fedavg_runner \
  --dataset {dataset} \
  --num_clients {num_clients} \
  --partition dirichlet \
  --dirichlet_alpha {dirichlet_alpha} \
  --use_encryption \
  --encryption_scheme paillier \
  --use_dp \
  --dp_mechanism {dp_mechanism} \
  --dp_clip_strategy fixed \
  --dp_clip_norm {dp_clip_norm} \
  --dp_epsilon {dp_epsilon}

/usr/bin/python3: Error while finding module specification for 'src.fl.fedavg_runner' (ModuleNotFoundError: No module named 'src')


**Strategy 2: Quantile Clipping (The Outlier Tamer)**

Concept: Instead of guessing a static limit, the algorithm looks at the actual distribution of gradient norms inside the local client's batches. It dynamically picks the norm at a target --dp_clip_quantile (e.g., the 50th percentile/median) and clips everything above it.

The 'Why': Quantile clipping perfectly handles Dirichlet non-IID spikes. It dynamically isolates the extreme outliers caused by skewed data batches while leaving the majority of the updates intact. It scales natively regardless of the dataset.

Execution: Quantile Clipping

In [ ]:
#@title Launch Quantile Clipping DP-FL
#@markdown This dynamically computes the clipping threshold based on the selected percentile of gradient norms.
dataset = "cifar10" #@param ["mnist", "cifar10", "ptbxl"]
dp_clip_quantile = 75.0 #@param {type:"slider", min:10.0, max:100.0, step:5.0}
dp_clip_min = 0.1 #@param {type:"number"}
dp_clip_max = 10.0 #@param {type:"number"}

!python -m src.fl.fedavg_runner \
  --dataset {dataset} \
  --num_clients 5 \
  --partition dirichlet \
  --use_encryption \
  --use_dp \
  --dp_clip_strategy quantile \
  --dp_clip_quantile {dp_clip_quantile} \
  --dp_clip_min {dp_clip_min} \
  --dp_clip_max {dp_clip_max}

**Strategy 3: Adaptive Clipping (The Convergence Tracker)**

Concept: Adaptive clipping operates like a moving average. Controlled by the --dp_clip_alpha (smoothing factor), the client tracks the moving average of unclipped norms over time.

The 'Why': In Federated Learning, gradients are huge in Round 1, but approach zero by Round 50 as the model converges. A threshold that works in Round 1 is too large for Round 50. Adaptive clipping decays the threshold in tandem with model convergence, ensuring an optimal signal-to-noise ratio without requiring you to manually schedule the threshold.

Execution: Adaptive Clipping

In [ ]:
#@title Launch Adaptive Clipping DP-FL
#@markdown This automatically adjusts the threshold over time based on the exponential moving average of gradient norms.

dataset = "ptbxl" #@param ["mnist", "cifar10", "ptbxl"]
ptbxl_model = "cnn_medium" #@param ["logistic", "cnn_medium", "cnn_large", "lstm"]
dp_clip_alpha = 0.9 #@param {type:"slider", min:0.1, max:0.99, step:0.01}

!python -m src.fl.fedavg_runner \
  --dataset {dataset} \
  --ptbxl_model {ptbxl_model} \
  --num_clients 5 \
  --use_encryption \
  --encryption_scheme ckks \
  --use_dp \
  --dp_clip_strategy adaptive \
  --dp_clip_alpha {dp_clip_alpha}